# Assignment 06: Clinic Data Wrangling Pipeline (Midterm)

Follow `README.md`: it names every file to save in `output/`, with its first line and line count. Replace each **TODO**, then restart the kernel and **Run All** before you commit, so the saved outputs show your work.

In [1]:
# Supplied setup: run this cell first and leave it unchanged.
from pathlib import Path
import hashlib
import json
import subprocess
import sys

import numpy as np
import pandas as pd

print("Python", sys.version.split()[0], "| NumPy", np.__version__, "| pandas", pd.__version__)


def find_assignment_dir(start):
    """The folder holding data_manifest.json: this one, a parent, or 06/mid-term below one."""
    for folder in (start, *start.parents):
        for candidate in (folder, folder / "06" / "mid-term"):
            if (candidate / "data_manifest.json").is_file():
                return candidate
    raise FileNotFoundError("Open this notebook from the assignment folder that holds data_manifest.json.")


BASE_DIR = find_assignment_dir(Path.cwd().resolve())
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)
PROFILE_PATH = OUTPUT_DIR / "table_profile.csv"
AUDIT_PATH = OUTPUT_DIR / "issue_audit.csv"
PATIENTS_SUMMARY_PATH = OUTPUT_DIR / "patients_summary.csv"
GENDER_CATEGORIES_PATH = OUTPUT_DIR / "gender_categories.csv"
VISITS_SUMMARY_PATH = OUTPUT_DIR / "visits_summary.csv"
DECISION_LOG_PATH = OUTPUT_DIR / "decision_log.csv"
A1C_SUMMARY_PATH = OUTPUT_DIR / "a1c_summary.csv"
MERGE_AUDIT_PATH = OUTPUT_DIR / "merge_audit.csv"
SBP_WIDE_SAMPLE_PATH = OUTPUT_DIR / "sbp_wide_sample.csv"
ANALYSIS_SAMPLE_PATH = OUTPUT_DIR / "analysis_sample.csv"

# The patients whose rows the two extracts hold (README Tasks 4.3 and 5.2).
SAMPLE_IDS = ["P00001", "P00004", "P00088", "P00260", "P00446", "P00487", "P00631", "P00851",
              "P02354", "P03836", "P05763", "P07510", "P14287", "P16486", "P17390", "P30001"]

# The data is generated, not committed: generate_data.py writes the same tables on every computer.
manifest = json.loads((BASE_DIR / "data_manifest.json").read_text(encoding="utf-8"))
if not all((DATA_DIR / name).is_file() for name in manifest["files"]):
    print(subprocess.run([sys.executable, "generate_data.py"], cwd=BASE_DIR, check=True,
                         capture_output=True, text=True).stdout.strip())


def read_raw(path):
    """A Parquet file with the dtypes it stores; a CSV file with every column as text, blanks kept as written."""
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    return pd.read_csv(path, dtype="string", keep_default_na=False)


raw = {Path(name).stem: read_raw(DATA_DIR / name) for name in manifest["files"]}
# The checksum covers each table's contents, written out as CSV text.
for name, expected in manifest["files"].items():
    text = raw[Path(name).stem].to_csv(index=False, lineterminator="\n")
    assert hashlib.sha256(text.encode("utf-8")).hexdigest() == expected["sha256"], (
        f"data/{name} differs from the generated file; delete the data folder and run this cell again.")
print(f"Verified {len(manifest['files'])} data files")
raw_snapshot = {name: table.copy(deep=True) for name, table in raw.items()}
list(raw)

Python 3.10.18 | NumPy 2.2.6 | pandas 2.3.3


Verified 5 data files


['patients', 'sites', 'visits_north', 'visits_south', 'a1c_wide']

## Cumulative midterm checkpoint (human review)

- **Lecture 01:** Task 1.1 uses the shell command `wc -l data/*.csv > output/file_inventory.txt` to count CSV lines, including their headers, and redirect the result to a file.
- **Lecture 02:** Task 1.2 defines `profile(name, table)` and loops over the manifest to save one dictionary per input file in `table_profile.csv`; the repository records this work with Git.
- **Lecture 03:** Task 4.3 converts the complete baseline/week12 pairs to a NumPy array and uses `axis=0` to reduce across patient rows, retaining two columns and therefore one mean per visit; the array shape and means are saved below that task.
- **Lecture 04:** Task 1.2 constructs a pandas DataFrame for table profiling; Tasks 5.2 and 5.3 save and reload the analysis extract with explicit dtypes and verify equality.
- **Lecture 05:** Tasks 2 and 3 apply a data contract, audit untouched inputs, map spelling variants, parse only exact calendar dates, and retain the later recheck; Task 3.1 converts gender to category and exports its category codes, and Task 5 asserts named validation checks.
- **Lecture 06:** Task 2.2 concatenates site exports, Task 4.1 melts A1c results, Task 4.3 pivots blood pressure, and Task 4.4 uses validated left joins to preserve every clean visit.

## Task 1: Inventory the files

### 1.1 Count lines from the terminal

Make `output/file_inventory.txt` in the terminal (README Task 1.1), then paste the command you ran into the `%%bash` cell below.

In [2]:
# On Windows, use Git Bash explicitly rather than the Windows WSL launcher.
import os
from IPython.core.magics.script import ScriptMagics

if sys.platform == "win32":
    git_bin = Path(os.environ.get("ProgramFiles", "C:/Program Files")) / "Git" / "bin"
    if (git_bin / "bash.exe").is_file():
        shell = get_ipython()
        shell.config.ScriptMagics.script_paths = {"bash": '"' + (git_bin / "bash.exe").as_posix() + '"'}
        shell.register_magics(ScriptMagics)

In [3]:
%%bash
wc -l data/*.csv > output/file_inventory.txt
cat output/file_inventory.txt

      6 data/sites.csv
  26298 data/visits_north.csv
  31075 data/visits_south.csv
  57379 total


### 1.2 Profile each table with a function

In [4]:
def profile(name, table):
    """Return the size and exact-repeat count of one raw input table."""
    return {
        "table": name,
        "rows": len(table),
        "columns": len(table.columns),
        "exact_duplicate_rows": int(table.duplicated().sum()),
    }


profiles = [profile(name, raw[Path(name).stem]) for name in manifest["files"]]
table_profile = pd.DataFrame(profiles)
table_profile.to_csv(PROFILE_PATH, index=False)
table_profile

,table,rows,columns,exact_duplicate_rows
0,patients.parquet,30045,4,45
1,sites.csv,5,3,0
2,visits_north.csv,26297,6,35
3,visits_south.csv,31074,6,35
4,a1c_wide.parquet,29424,3,0


## Task 2: Contract, stack, and audit

### 2.1 Write the data contract (human review)

| Cleaned input | Intended grain | Intended key |
| --- | --- | --- |
| Patients | One registered patient after removing exact re-exported rows | `patient_id` (single) |
| Sites | One current directory record per site; retired versions are excluded | `site_code` (single) |
| North visits | One patient's coded visit from the north export after stacking with the south export, removing exact repeats, and keeping the later recheck | `patient_id`, `visit` (composite in the combined visits table) |
| South visits | One patient's coded visit from the south export under the same stacking and later-recheck rule | `patient_id`, `visit` (composite in the combined visits table) |
| Wide A1c | One patient's central lab record, with separate baseline and week12 columns | `patient_id` (single) |

Raw patient re-exports and exact visit repeats violate these intended cleaned keys. Rechecks also repeat a patient/visit key but differ in time or measurements; mapping visit labels must precede checking that key. Current and retired site versions share a `site_code` without being exact duplicates. The raw A1c patient key is unique. After melting and dropping absent results, `a1c_long` has one available result per `patient_id`, `visit` pair.

The Task 4.4 joins are **many-to-one** from visits to patients on `patient_id`, **many-to-one** from the patient information carried on visits to current sites on `site_code`, and **one-to-one** from clean visits to long A1c results on `patient_id`, `visit`. All three use left joins so unmatched visits survive with missing lookup values. The code checks the stated cardinality with `validate=`.

### 2.2 Stack the two visit files

In [5]:
north = raw["visits_north"].copy(deep=True)
south = raw["visits_south"].copy(deep=True).rename(columns={"wt_kg": "weight_kg"})
north["source_file"] = "visits_north.csv"
south["source_file"] = "visits_south.csv"
visits_raw = pd.concat([north, south], ignore_index=True)
visits_raw_snapshot = visits_raw.copy(deep=True)
assert visits_raw.shape == (len(north) + len(south), 7)
assert not visits_raw.isna().all().any()
print("Stacked visits:", visits_raw.shape)
visits_raw.head()

Stacked visits: (57371, 7)


,patient_id,visit,visit_date,visit_time,sbp_mmhg,weight_kg,source_file
0,P00002,baseline,2026-02-20,09:50,132,85.2,visits_north.csv
1,P00002,week12,2026-05-08,15:20,129,82.0,visits_north.csv
2,P00003,baseline,2026-01-16,11:45,148,67.5,visits_north.csv
3,P00003,wk12,2026-04-07,10:00,137,64.0,visits_north.csv
4,P00004,baseline,2026-02-25,10:55,145,106.2,visits_north.csv


### 2.3 Audit the raw tables

In [6]:
VISIT_MAP = {
    "baseline": "baseline", "bl": "baseline",
    "week12": "week12", "week 12": "week12", "wk12": "week12",
}


def exact_dates(values):
    """Require both exact YYYY-MM-DD text and a real calendar date."""
    exact_text = values.str.fullmatch(r"[0-9]{4}-[0-9]{2}-[0-9]{2}", na=False)
    return pd.to_datetime(values.where(exact_text), format="%Y-%m-%d", errors="coerce")


patients_raw = raw["patients"]
gender_empty = patients_raw["gender"].str.strip().eq("")
normalized_sites = patients_raw["site_code"].str.upper().str.replace("-", "", regex=False).str.replace(" ", "", regex=False)
sbp_raw = pd.to_numeric(visits_raw["sbp_mmhg"], errors="coerce")
visit_keys = visits_raw[["patient_id", "visit"]].copy(deep=True)
visit_keys["visit"] = visit_keys["visit"].str.strip().str.lower().map(VISIT_MAP)

issue_audit = pd.DataFrame({
    "issue": [
        "patients: gender values that are not a code",
        "patients: empty gender",
        "patients: birth_year sentinel 9999",
        "patients: site codes needing normalization",
        "sites: rows with a repeated site_code",
        "visits: visit labels that are not baseline or week12",
        "visits: sbp sentinel 999",
        "visits: sbp outside 70 through 250",
        "visits: date parse failures",
        "visits: rows in repeated patient_id and visit pairs",
    ],
    "count": [
        ((~gender_empty) & ~patients_raw["gender"].isin(["F", "M", "X"])).sum(),
        gender_empty.sum(),
        patients_raw["birth_year"].eq("9999").sum(),
        patients_raw["site_code"].ne(normalized_sites).sum(),
        raw["sites"].duplicated(subset=["site_code"], keep=False).sum(),
        (~visits_raw["visit"].isin(["baseline", "week12"])).sum(),
        sbp_raw.eq(999).sum(),
        (sbp_raw.notna() & sbp_raw.ne(999) & ~sbp_raw.between(70, 250)).sum(),
        exact_dates(visits_raw["visit_date"]).isna().sum(),
        visit_keys.duplicated(subset=["patient_id", "visit"], keep=False).sum(),
    ],
})
issue_audit["count"] = issue_audit["count"].astype("int64")
issue_audit.to_csv(AUDIT_PATH, index=False)
display(issue_audit)
print("Raw tables unchanged:", all(raw[name].equals(raw_snapshot[name]) for name in raw))
assert all(raw[name].equals(raw_snapshot[name]) for name in raw), "Auditing must not change raw."

,issue,count
0,patients: gender values that are not a code,6968
1,patients: empty gender,486
2,patients: birth_year sentinel 9999,371
3,patients: site codes needing normalization,1811
4,sites: rows with a repeated site_code,4
5,visits: visit labels that are not baseline or ...,2309
6,visits: sbp sentinel 999,341
7,visits: sbp outside 70 through 250,116
8,visits: date parse failures,169
9,visits: rows in repeated patient_id and visit ...,380


Raw tables unchanged: True


## Task 3: Clean

### 3.1 Clean the patients

In [7]:
display(raw["patients"]["gender"].value_counts(dropna=False))
GENDER_MAP = {
    "f": "F", "female": "F", "woman": "F",
    "m": "M", "male": "M", "man": "M",
    "x": "X", "nonbinary": "X", "non-binary": "X", "nb": "X",
}
patients_clean = raw["patients"].copy(deep=True).drop_duplicates(keep="first")
patients_clean["gender"] = patients_clean["gender"].str.strip().str.lower().map(GENDER_MAP).astype("category")
patients_clean["birth_year"] = pd.to_numeric(
    patients_clean["birth_year"].replace("9999", pd.NA), errors="coerce"
).astype("Int64")
patients_clean["site_code"] = patients_clean["site_code"].str.upper().str.replace("-", "", regex=False).str.replace(" ", "", regex=False)
patients_clean = patients_clean.reset_index(drop=True)
patients_clean.head()

gender
F             11248
M             10971
male           1039
female         1018
Male            622
f               622
Female          578
m               538
woman           400
                382
X               372
man             329
unknown         282
MALE            256
FEMALE          231
 M              223
 F              205
Woman           131
nonbinary       118
NB              104
                104
Unknown          96
non-binary       80
x                62
Non-Binary       34
Name: count, dtype: Int64

,patient_id,gender,birth_year,site_code
0,P00001,F,1966,S02
1,P00002,F,<NA>,N01
2,P00003,F,1970,N01
3,P00004,X,<NA>,N01
4,P00005,M,1947,N01


In [8]:
display(patients_clean["gender"].cat.categories)
patients_summary = pd.DataFrame({
    "metric": ["patients", "birth_year missing", "birth_year min", "birth_year max",
               "site_code values", "patients at N01", "patients at S02", "patients at E04"],
    "value": [len(patients_clean), patients_clean["birth_year"].isna().sum(),
              patients_clean["birth_year"].min(), patients_clean["birth_year"].max(),
              patients_clean["site_code"].nunique(), patients_clean["site_code"].eq("N01").sum(),
              patients_clean["site_code"].eq("S02").sum(), patients_clean["site_code"].eq("E04").sum()],
})
patients_summary.to_csv(PATIENTS_SUMMARY_PATH, index=False)
gender_counts = patients_clean["gender"].value_counts(dropna=False).rename_axis("gender").reset_index(name="count")
gender_counts["code"] = gender_counts["gender"].cat.codes
gender_counts.to_csv(GENDER_CATEGORIES_PATH, index=False)
display(patients_summary, gender_counts)

Index(['F', 'M', 'X'], dtype='object')

,metric,value
0,patients,30000
1,birth_year missing,371
2,birth_year min,1941
3,birth_year max,1996
4,site_code values,3
5,patients at N01,13739
6,patients at S02,15706
7,patients at E04,555


,gender,count,code
0,F,14413,0
1,M,13955,1
2,NaN,863,-1
3,X,769,2


### 3.2 Clean the visits

In [9]:
display(visits_raw["visit"].value_counts(dropna=False))
visits_clean = visits_raw.copy(deep=True).drop_duplicates(keep="first")
visits_clean["visit"] = visits_clean["visit"].str.strip().str.lower().map(VISIT_MAP)
sbp = pd.to_numeric(visits_clean["sbp_mmhg"], errors="coerce")
visits_clean["sbp_mmhg"] = sbp.where(sbp.between(70, 250)).astype("Int64")
visits_clean["weight_kg"] = pd.to_numeric(visits_clean["weight_kg"], errors="coerce").astype("float64")
visits_clean["visit_date"] = exact_dates(visits_clean["visit_date"])
assert visits_clean["visit"].notna().all()

visit
baseline     28533
week12       26529
BASELINE       256
Baseline       242
bl             235
baseline       228
BL             220
wk12           204
 week12        198
Week 12        195
WEEK12         186
Wk12           177
week 12        168
Name: count, dtype: Int64

In [10]:
rechecks = visits_clean.loc[
    visits_clean.duplicated(subset=["patient_id", "visit"], keep=False)
].sort_values(["patient_id", "visit", "visit_time"])
print("Rows in recheck pairs:", len(rechecks))
display(rechecks)
visits_clean = visits_clean.sort_values("visit_time", kind="stable").drop_duplicates(
    subset=["patient_id", "visit"], keep="last"
)

Rows in recheck pairs: 240


,patient_id,visit,visit_date,visit_time,sbp_mmhg,weight_kg,source_file
26298,P00001,baseline,2026-03-06,14:15,141,52.4,visits_south.csv
26297,P00001,baseline,2026-03-06,14:40,146,52.4,visits_south.csv
4,P00004,baseline,2026-02-25,10:55,145,106.2,visits_north.csv
5,P00004,baseline,2026-02-25,11:15,142,106.2,visits_north.csv
84,P00088,baseline,2026-03-10,09:30,153,68.2,visits_north.csv
...,...,...,...,...,...,...,...
55473,P28232,baseline,2026-01-12,10:10,159,61.0,visits_south.csv
56249,P28953,baseline,2026-01-21,10:30,153,74.8,visits_south.csv
56248,P28953,baseline,2026-01-21,10:45,151,74.8,visits_south.csv
25464,P29170,baseline,2026-03-02,11:20,130,78.0,visits_north.csv


In [11]:
visits_clean = visits_clean.sort_values(["patient_id", "visit"]).reset_index(drop=True)[
    ["patient_id", "visit", "visit_date", "sbp_mmhg", "weight_kg", "source_file"]
]
display(visits_clean["visit"].value_counts(dropna=False))
visits_summary = pd.DataFrame({
    "metric": ["visits", "baseline visits", "week12 visits", "visit_date missing", "visit_date min",
               "visit_date max", "sbp_mmhg missing", "sbp_mmhg min", "sbp_mmhg max", "weight_kg missing",
               "weight_kg mean", "visits from visits_north.csv", "visits from visits_south.csv"],
    "value": [len(visits_clean), visits_clean["visit"].eq("baseline").sum(),
              visits_clean["visit"].eq("week12").sum(), visits_clean["visit_date"].isna().sum(),
              visits_clean["visit_date"].min(), visits_clean["visit_date"].max(),
              visits_clean["sbp_mmhg"].isna().sum(), visits_clean["sbp_mmhg"].min(),
              visits_clean["sbp_mmhg"].max(), visits_clean["weight_kg"].isna().sum(),
              round(visits_clean["weight_kg"].mean(), 2),
              visits_clean["source_file"].eq("visits_north.csv").sum(),
              visits_clean["source_file"].eq("visits_south.csv").sum()],
})
visits_summary.to_csv(VISITS_SUMMARY_PATH, index=False)
visits_summary

visit
baseline    29599
week12      27582
Name: count, dtype: int64

,metric,value
0,visits,57181
1,baseline visits,29599
2,week12 visits,27582
3,visit_date missing,169
4,visit_date min,2026-01-05 00:00:00
5,visit_date max,2026-07-06 00:00:00
6,sbp_mmhg missing,456
7,sbp_mmhg min,70
8,sbp_mmhg max,250
9,weight_kg missing,828


### 3.3 Record the decisions

In [12]:
decision_table = pd.DataFrame([
    ("patients", "exact duplicate rows", "Remove exact repeats and keep the first row.", "The registry re-export repeats identical patient records rather than recording additional patients."),
    ("patients", "gender spellings that are not a code", "Strip and lowercase text, map known spellings to F/M/X, and leave blanks and unknown missing.", "Female/woman, male/man, and nonbinary/NB spellings identify the dictionary categories while unknown supplies no gender."),
    ("patients", "site codes needing normalization", "Uppercase site codes and remove every space and hyphen.", "Spellings such as N-01 and n01 refer to N01, while E04 must remain a separate registry code even though the directory omits it."),
    ("patients", "birth_year sentinel 9999", "Replace 9999 with missing and use nullable Int64.", "The data dictionary defines 9999 as not recorded rather than a patient's birth year."),
    ("visits", "weight column named wt_kg in visits_south.csv", "Rename wt_kg to weight_kg before stacking and preserve the exporting filename.", "Both offices report the same kilogram measurement, and the south export also serves E04 patients."),
    ("visits", "sbp sentinel 999 and readings outside 70 through 250", "Convert SBP to numbers and retain only the inclusive 70 to 250 range as nullable Int64.", "999 means not measured and other out-of-range entries violate the contract, while the planted 70 and 250 readings satisfy it."),
    ("visits", "impossible and loosely written visit dates", "Require exact YYYY-MM-DD text and a real calendar date; otherwise store NaT.", "Dates such as 2026-02-30 are impossible and 2026/02/02 fails the required input format."),
    ("visits", "repeated patient_id and visit pairs", "Remove exact repeats, normalize visit labels, then retain the reading with the later visit_time.", "The remaining repeated pairs are same-visit rechecks taken 10 to 30 minutes later and should contribute one updated reading."),
    ("sites", "repeated site_code rows", "Keep current records only and select site_code and site_name.", "N01 and S02 each have a retired name, so using all versions would multiply patient rows and attach obsolete names."),
], columns=["table", "issue", "decision", "reason"])
decision_table.to_csv(DECISION_LOG_PATH, index=False)
decision_table

,table,issue,decision,reason
0,patients,exact duplicate rows,Remove exact repeats and keep the first row.,The registry re-export repeats identical patie...
1,patients,gender spellings that are not a code,"Strip and lowercase text, map known spellings ...","Female/woman, male/man, and nonbinary/NB spell..."
2,patients,site codes needing normalization,Uppercase site codes and remove every space an...,"Spellings such as N-01 and n01 refer to N01, w..."
3,patients,birth_year sentinel 9999,Replace 9999 with missing and use nullable Int64.,The data dictionary defines 9999 as not record...
4,visits,weight column named wt_kg in visits_south.csv,Rename wt_kg to weight_kg before stacking and ...,Both offices report the same kilogram measurem...
5,visits,sbp sentinel 999 and readings outside 70 throu...,Convert SBP to numbers and retain only the inc...,999 means not measured and other out-of-range ...
6,visits,impossible and loosely written visit dates,Require exact YYYY-MM-DD text and a real calen...,Dates such as 2026-02-30 are impossible and 20...
7,visits,repeated patient_id and visit pairs,"Remove exact repeats, normalize visit labels, ...",The remaining repeated pairs are same-visit re...
8,sites,repeated site_code rows,Keep current records only and select site_code...,"N01 and S02 each have a retired name, so using..."


## Task 4: Reshape and join

### 4.1 Melt the A1c results to long form

In [13]:
a1c_long = raw["a1c_wide"].copy(deep=True).melt(
    id_vars=["patient_id"], value_vars=["a1c_baseline", "a1c_week12"],
    var_name="visit", value_name="a1c_pct",
)
a1c_long["visit"] = a1c_long["visit"].map({"a1c_baseline": "baseline", "a1c_week12": "week12"})
print("A1c dtype:", a1c_long["a1c_pct"].dtype)
assert a1c_long["a1c_pct"].dtype == np.dtype("float64")
a1c_long = a1c_long.dropna(subset=["a1c_pct"]).reset_index(drop=True)
a1c_summary = pd.DataFrame([
    {"visit": visit, "results": int(a1c_long["visit"].eq(visit).sum()),
     "a1c_pct_mean": round(a1c_long.loc[a1c_long["visit"].eq(visit), "a1c_pct"].mean(), 2)}
    for visit in ["baseline", "week12"]
])
a1c_summary.to_csv(A1C_SUMMARY_PATH, index=False)
a1c_summary

A1c dtype: float64


,visit,results,a1c_pct_mean
0,baseline,29107,8.15
1,week12,27653,7.63


### 4.2 Check the keys before joining

In [14]:
sites = raw["sites"]
try:
    patients_clean.merge(sites, on="site_code", how="left", validate="many_to_one")
except pd.errors.MergeError as error:
    print("Expected MergeError:", error)

display(sites.loc[sites.duplicated(subset=["site_code"], keep=False)].sort_values("site_code"))
sites_current = sites.loc[sites["record_status"].eq("current"), ["site_code", "site_name"]].copy(deep=True)
assert sites_current["site_code"].is_unique
sites_current

Expected MergeError: Merge keys are not unique in right dataset; not a many-to-one merge


,site_code,site_name,record_status
0,N01,North Clinic,current
1,N01,North Street Clinic,retired
2,S02,Southgate Clinic,retired
3,S02,South Valley Clinic,current


,site_code,site_name
0,N01,North Clinic
3,S02,South Valley Clinic
4,W03,West Clinic


In [15]:
visit_lab_audit = visits_clean.merge(
    a1c_long, on=["patient_id", "visit"], how="outer", indicator=True, validate="one_to_one"
)
merge_counts = visit_lab_audit["_merge"].value_counts()
display(merge_counts)
merge_audit = pd.DataFrame({
    "check": ["visits with an A1c result", "visits without an A1c result", "A1c results without a visit",
              "visits whose patient is not in the registry", "registry patients with no visit",
              "registry patients whose site is not in the site list", "site list rows removed by the current-record rule"],
    "count": [merge_counts.get("both", 0), merge_counts.get("left_only", 0), merge_counts.get("right_only", 0),
              (~visits_clean["patient_id"].isin(patients_clean["patient_id"])).sum(),
              (~patients_clean["patient_id"].isin(visits_clean["patient_id"])).sum(),
              (~patients_clean["site_code"].isin(sites["site_code"])).sum(), len(sites) - len(sites_current)],
})
merge_audit["count"] = merge_audit["count"].astype("int64")
merge_audit.to_csv(MERGE_AUDIT_PATH, index=False)
merge_audit

_merge
both          54006
left_only      3175
right_only     2754
Name: count, dtype: int64

,check,count
0,visits with an A1c result,54006
1,visits without an A1c result,3175
2,A1c results without a visit,2754
3,visits whose patient is not in the registry,79
4,registry patients with no visit,441
5,registry patients whose site is not in the sit...,555
6,site list rows removed by the current-record rule,2


### 4.3 Pivot blood pressure to one row per patient

In [16]:
sbp_wide = visits_clean.pivot(index="patient_id", columns="visit", values="sbp_mmhg").reset_index()
sbp_wide.columns.name = None
sbp_wide["sbp_change"] = sbp_wide["week12"] - sbp_wide["baseline"]
sbp_wide_sample = sbp_wide.loc[sbp_wide["patient_id"].isin(SAMPLE_IDS)]
sbp_wide_sample.to_csv(SBP_WIDE_SAMPLE_PATH, index=False)
sbp_wide_sample

,patient_id,baseline,week12,sbp_change
0,P00001,146,129,-17
3,P00004,142,<NA>,<NA>
87,P00088,152,147,-5
259,P00260,70,138,68
445,P00446,250,144,-106
486,P00487,<NA>,143,<NA>
849,P00851,<NA>,129,<NA>
2336,P02354,127,125,-2
3789,P03836,134,125,-9
5684,P05763,161,151,-10


#### NumPy checkpoint (Lecture 03, human review)

From the full `sbp_wide`, select `baseline` and `week12` in that order, keeping only patients with both readings. Name this two-column table `complete_pairs`; `np.array(complete_pairs, dtype=float)` converts it to a NumPy array. Display its shape and calculate one NumPy mean per visit across patients. Save the outputs and explain your axis choice in the cumulative checkpoint's Lecture 03 bullet. No extra output file.

In [17]:
complete_pairs = sbp_wide[["baseline", "week12"]].dropna()
bp_array = np.array(complete_pairs, dtype=float)
print("Complete-pair array shape:", bp_array.shape)
print("Mean SBP [baseline, week12] (mmHg):", np.mean(bp_array, axis=0))

Complete-pair array shape: (27149, 2)
Mean SBP [baseline, week12] (mmHg): [143.01337066 137.15061328]


### 4.4 Join the analysis table

In [18]:
analysis = visits_clean.merge(patients_clean, on="patient_id", how="left", validate="many_to_one")
assert len(analysis) == len(visits_clean)
analysis = analysis.merge(sites_current, on="site_code", how="left", validate="many_to_one")
assert len(analysis) == len(visits_clean)
analysis = analysis.merge(a1c_long, on=["patient_id", "visit"], how="left", validate="one_to_one")
assert len(analysis) == len(visits_clean)
analysis_dtypes = {
    "patient_id": "string", "visit": "string", "visit_date": "datetime64[us]",
    "sbp_mmhg": "Int64", "weight_kg": "float64", "a1c_pct": "float64",
    "gender": "string", "birth_year": "Int64", "site_code": "string",
    "site_name": "string", "source_file": "string",
}
analysis = analysis[list(analysis_dtypes)].astype(analysis_dtypes)
analysis.dtypes

patient_id     string[python]
visit          string[python]
visit_date     datetime64[us]
sbp_mmhg                Int64
weight_kg             float64
a1c_pct               float64
gender         string[python]
birth_year              Int64
site_code      string[python]
site_name      string[python]
source_file    string[python]
dtype: object

## Task 5: Validate, save, and read back

### 5.1 Validate before saving

In [19]:
checks = pd.Series({
    "analysis preserves every clean visit": len(analysis) == len(visits_clean),
    "analysis patient/visit keys are unique": not analysis.duplicated(["patient_id", "visit"]).any(),
    "present SBP is in the inclusive range": analysis["sbp_mmhg"].dropna().between(70, 250).all(),
    "present site names are current": analysis["site_name"].dropna().isin(sites_current["site_name"]).all(),
    "wide SBP patient IDs are unique": sbp_wide["patient_id"].is_unique,
    "raw inputs are unchanged": all(raw[name].equals(raw_snapshot[name]) for name in raw),
    "stacked raw visits are unchanged": visits_raw.equals(visits_raw_snapshot),
    "registry patient IDs are unique": patients_clean["patient_id"].is_unique,
    "long A1c keys are unique": not a1c_long.duplicated(["patient_id", "visit"]).any(),
    "visit labels match the contract": analysis["visit"].isin(["baseline", "week12"]).all(),
    "analysis dtypes match the contract": all(str(analysis[column].dtype) == dtype for column, dtype in analysis_dtypes.items()),
}, dtype=bool)
display(checks)
assert checks.all(), checks[~checks]

analysis preserves every clean visit      True
analysis patient/visit keys are unique    True
present SBP is in the inclusive range     True
present site names are current            True
wide SBP patient IDs are unique           True
raw inputs are unchanged                  True
stacked raw visits are unchanged          True
registry patient IDs are unique           True
long A1c keys are unique                  True
visit labels match the contract           True
analysis dtypes match the contract        True
dtype: bool

### 5.2 Save the analysis extract

In [20]:
analysis_sample = analysis.loc[analysis["patient_id"].isin(SAMPLE_IDS)].reset_index(drop=True)
analysis_sample.to_csv(ANALYSIS_SAMPLE_PATH, index=False)
print("Sample visits:", len(analysis_sample))
analysis_sample

Sample visits: 28


,patient_id,visit,visit_date,sbp_mmhg,weight_kg,a1c_pct,gender,birth_year,site_code,site_name,source_file
0,P00001,baseline,2026-03-06,146,52.4,7.3,F,1966,S02,South Valley Clinic,visits_south.csv
1,P00001,week12,2026-05-27,129,50.2,NaN,F,1966,S02,South Valley Clinic,visits_south.csv
2,P00004,baseline,2026-02-25,142,106.2,7.3,X,<NA>,N01,North Clinic,visits_north.csv
3,P00088,baseline,2026-03-10,152,68.2,8.4,F,1980,N01,North Clinic,visits_north.csv
4,P00088,week12,2026-06-03,147,66.3,7.6,F,1980,N01,North Clinic,visits_north.csv
5,P00260,baseline,NaT,70,80.4,9.5,F,1979,E04,<NA>,visits_south.csv
6,P00260,week12,2026-06-10,138,78.6,8.0,F,1979,E04,<NA>,visits_south.csv
7,P00446,baseline,NaT,250,NaN,6.5,F,1946,S02,South Valley Clinic,visits_south.csv
8,P00446,week12,2026-04-02,144,94.3,6.8,F,1946,S02,South Valley Clinic,visits_south.csv
9,P00487,baseline,2026-03-02,<NA>,103.0,7.9,M,1959,S02,South Valley Clinic,visits_south.csv


### 5.3 Read the file back

In [21]:
round_trip = pd.read_csv(
    ANALYSIS_SAMPLE_PATH,
    dtype={column: dtype for column, dtype in analysis_dtypes.items() if column != "visit_date"},
    parse_dates=["visit_date"],
)
# pandas 2.x parses dates at nanosecond resolution; match the contract's microseconds.
round_trip["visit_date"] = round_trip["visit_date"].astype("datetime64[us]")
print(round_trip.equals(analysis_sample))
assert round_trip.equals(analysis_sample), "CSV read-back must preserve values and dtypes."

True
